# BitNet b1.58 — YaRN Context Extension (4K → 8K)
**Sanity-check run:** 200 steps on free T4. Goal: confirm 8K loss decreases.  
If it does → run full 1000 steps on RTX 4090.

---


## 1. Install dependencies

In [ ]:
!pip install -q transformers==4.46.3 accelerate datasets safetensors bitsandbytes

## 2. Imports & config

In [ ]:
import os, math, time, types
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

# ── Config ─────────────────────────────────────────────────────────────────
REPO_ID       = "microsoft/bitnet-b1.58-2B-4T-bf16"
ORIGINAL_MAX  = 4096
TARGET_MAX    = 8192
ROPE_THETA    = 500_000
HEAD_DIM      = 128
SCALE_ZONE    = (32, 42)   # bands to scale (from forensic audit)
SCALE_FACTOR  = 2.0        # wavelength multiplier

LR            = 1e-5
TOTAL_STEPS   = 200
LOG_EVERY     = 10
SAVE_EVERY    = 100
BATCH_SIZE    = 1
GRAD_ACCUM    = 8          # effective batch = 8
MAX_SEQ_LEN   = 8192

DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE         = torch.bfloat16

print(f"Device : {DEVICE}")
print(f"CUDA   : {torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'N/A'}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB" if DEVICE=="cuda" else "")

## 3. Load bf16 model from HuggingFace

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(REPO_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model (bf16)... this takes ~2-3 minutes")
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    REPO_ID,
    torch_dtype=DTYPE,
    device_map="auto",
    low_cpu_mem_usage=True,
)
print(f"Loaded in {time.time()-t0:.1f}s")
print(f"Params : {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

# Confirm sub_norm present
sub_norm_count = sum(1 for n,_ in model.named_parameters() if 'sub_norm' in n)
print(f"Sub_norm tensors found: {sub_norm_count}  (expect 60)")

## 4. YaRN RoPE patch — selective band interpolation

In [ ]:
def compute_yarn_inv_freq(theta, head_dim, scale_zone, scale_factor):
    """
    Selective YaRN: only scale RoPE bands in scale_zone.
    Bands 0-31:  saturated   -> DO NOT TOUCH
    Bands 32-42: transition  -> scale by 1/scale_factor (halve frequency)
    Bands 43-63: safe        -> DO NOT TOUCH
    """
    i = np.arange(0, head_dim, 2, dtype=np.float64)
    inv_freq = 1.0 / (theta ** (i / head_dim))
    
    scale_array = np.ones(head_dim // 2, dtype=np.float64)
    lo, hi = scale_zone
    scale_array[lo : hi + 1] = 1.0 / scale_factor  # halve frequency = double wavelength
    
    return (inv_freq * scale_array).astype(np.float32)

yarn_inv_freq = compute_yarn_inv_freq(ROPE_THETA, HEAD_DIM, SCALE_ZONE, SCALE_FACTOR)
yarn_inv_freq_tensor = torch.tensor(yarn_inv_freq, dtype=torch.float32)

# Patch every layer's rotary embedding
patched = 0
for layer in model.model.layers:
    attn = layer.self_attn
    if hasattr(attn, 'rotary_emb'):
        re = attn.rotary_emb
        # Replace inv_freq buffer
        re.register_buffer('inv_freq', yarn_inv_freq_tensor.clone().to(DEVICE))
        patched += 1

# Update max position embeddings in config
model.config.max_position_embeddings = TARGET_MAX
model.generation_config.max_length   = TARGET_MAX

print(f"Patched rotary_emb in {patched} layers  (expect 30)")

# Verify: spot check layer 0
sample = model.model.layers[0].self_attn.rotary_emb.inv_freq
wl_32_base = (2 * math.pi) / (1.0 / (ROPE_THETA ** (64 / HEAD_DIM)))
wl_32_yarn = (2 * math.pi) / sample[32].item()
print(f"Band 32 wavelength — base: {wl_32_base:.1f}  yarn: {wl_32_yarn:.1f}  ratio: {wl_32_yarn/wl_32_base:.3f}  (expect 2.000)")

## 5. Freeze strategy — layers 25-29 sub_norm (phase 1)

In [ ]:
HIGH_GAIN_FREEZE = list(range(25, 30))  # layers 25-29

def set_frozen(freeze_high_gain: bool):
    for name, param in model.named_parameters():
        if any(f'layers.{i}.' in name for i in HIGH_GAIN_FREEZE) and 'sub_norm' in name:
            param.requires_grad = not freeze_high_gain
        else:
            param.requires_grad = True
    
    frozen  = sum(p.numel() for n,p in model.named_parameters() if not p.requires_grad)
    trainable = sum(p.numel() for n,p in model.named_parameters() if p.requires_grad)
    print(f"Frozen   : {frozen/1e6:.2f}M params")
    print(f"Trainable: {trainable/1e6:.2f}M params")

print("Phase 1: freezing layers 25-29 sub_norm")
set_frozen(freeze_high_gain=True)

## 6. Dataset — PG19 long documents

In [ ]:
print("Loading PG19 train split (streaming)...")
dataset = load_dataset("deepmind/pg19", split="train", streaming=True, trust_remote_code=True)

def tokenize_long_docs(dataset, tokenizer, min_tokens=6000, chunk_size=8192, n_chunks=500):
    """
    Stream PG19, filter docs >6K tokens, chunk to exactly 8192 tokens.
    Returns list of input_ids tensors ready for training.
    """
    chunks = []
    buffer = []
    
    for doc in dataset:
        if len(chunks) >= n_chunks:
            break
        ids = tokenizer(doc['text'], add_special_tokens=False)['input_ids']
        if len(ids) < min_tokens:
            continue
        buffer.extend(ids)
        while len(buffer) >= chunk_size:
            chunks.append(torch.tensor(buffer[:chunk_size], dtype=torch.long))
            buffer = buffer[chunk_size:]
    
    print(f"Built {len(chunks)} chunks of {chunk_size} tokens")
    return chunks

chunks = tokenize_long_docs(dataset, tokenizer, n_chunks=500)
print(f"Dataset ready. Total tokens: {len(chunks) * MAX_SEQ_LEN / 1e6:.1f}M")

## 7. Baseline — PPL at 4K and 8K (pre-training)

In [ ]:
@torch.no_grad()
def compute_ppl(model, chunks, max_len, n_samples=20):
    """Compute perplexity on first n_samples chunks truncated to max_len."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    
    for chunk in chunks[:n_samples]:
        ids = chunk[:max_len].unsqueeze(0).to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=DTYPE):
            out = model(ids, labels=ids)
        total_loss   += out.loss.item() * (ids.shape[1] - 1)
        total_tokens += ids.shape[1] - 1
    
    avg_loss = total_loss / total_tokens
    return math.exp(avg_loss), avg_loss

print("Computing baseline PPL at 4K tokens...")
ppl_4k_base, loss_4k_base = compute_ppl(model, chunks, max_len=4096)
print(f"  PPL@4K  = {ppl_4k_base:.3f}  (loss={loss_4k_base:.4f})")

print("Computing baseline PPL at 8K tokens...")
ppl_8k_base, loss_8k_base = compute_ppl(model, chunks, max_len=8192)
print(f"  PPL@8K  = {ppl_8k_base:.3f}  (loss={loss_8k_base:.4f})")

print(f"\nBaseline degradation at 8K: {ppl_8k_base - ppl_4k_base:+.3f} PPL points")
print("(expect high — model has never seen 8K context)")

## 8. Training loop — 200 steps

In [ ]:
import random
from torch.optim import AdamW

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.01,
)

# Gradient checkpointing to save VRAM
model.gradient_checkpointing_enable()
model.train()

log = {
    'step': [], 'loss_4k': [], 'loss_8k': [], 'phase': []
}

print(f"Starting training — {TOTAL_STEPS} steps, lr={LR}")
print(f"Phase 1 (steps 0-99):   layers 25-29 sub_norm FROZEN")
print(f"Phase 2 (steps 100-199): layers 25-29 sub_norm UNFROZEN")
print("─" * 60)

optimizer.zero_grad()
t_start = time.time()

for step in range(TOTAL_STEPS):
    
    # ── Phase transition at step 100 ──────────────────────────────────────
    if step == 100:
        print("\n→ Phase 2: unfreezing layers 25-29 sub_norm")
        set_frozen(freeze_high_gain=False)
        # Rebuild optimizer with new param set
        optimizer = AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=LR * 0.5,   # halve lr for cautious unfreeze
            weight_decay=0.01,
        )
        optimizer.zero_grad()
    
    # ── Sample a random 8K chunk ──────────────────────────────────────────
    chunk = random.choice(chunks).to(DEVICE)
    ids   = chunk.unsqueeze(0)   # (1, 8192)
    
    # ── Forward + loss ────────────────────────────────────────────────────
    with torch.autocast(device_type='cuda', dtype=DTYPE):
        out = model(ids, labels=ids)
    loss = out.loss / GRAD_ACCUM
    loss.backward()
    
    # ── Gradient accumulation ─────────────────────────────────────────────
    if (step + 1) % GRAD_ACCUM == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()
    
    # ── Logging ───────────────────────────────────────────────────────────
    if step % LOG_EVERY == 0:
        elapsed = time.time() - t_start
        steps_left = TOTAL_STEPS - step
        eta = (elapsed / max(step, 1)) * steps_left
        
        # Quick loss estimate at 4K vs 8K (no grad, 5 samples)
        model.eval()
        with torch.no_grad():
            _, l4k = compute_ppl(model, chunks, max_len=4096, n_samples=5)
            _, l8k = compute_ppl(model, chunks, max_len=8192, n_samples=5)
        model.train()
        
        phase = 1 if step < 100 else 2
        log['step'].append(step)
        log['loss_4k'].append(l4k)
        log['loss_8k'].append(l8k)
        log['phase'].append(phase)
        
        print(f"Step {step:>3} | loss@4K={l4k:.4f} | loss@8K={l8k:.4f} | "
              f"gap={l8k-l4k:+.4f} | phase={phase} | ETA={eta/60:.1f}min")
    
    # ── Checkpoint ────────────────────────────────────────────────────────
    if (step + 1) % SAVE_EVERY == 0:
        ckpt_path = f"/content/bitnet_yarn_step{step+1}"
        model.save_pretrained(ckpt_path)
        tokenizer.save_pretrained(ckpt_path)
        print(f"  → Checkpoint saved: {ckpt_path}")

print("\nTraining complete.")

## 9. Post-training evaluation

In [ ]:
model.eval()

print("Post-training PPL:")
ppl_4k_after, loss_4k_after = compute_ppl(model, chunks, max_len=4096, n_samples=20)
ppl_8k_after, loss_8k_after = compute_ppl(model, chunks, max_len=8192, n_samples=20)

print(f"  PPL@4K  : {ppl_4k_base:.3f} → {ppl_4k_after:.3f}  ({ppl_4k_after-ppl_4k_base:+.3f})")
print(f"  PPL@8K  : {ppl_8k_base:.3f} → {ppl_8k_after:.3f}  ({ppl_8k_after-ppl_8k_base:+.3f})")
print()

# Success criteria
delta_4k = ppl_4k_after - ppl_4k_base
delta_8k = ppl_8k_after - ppl_8k_base

if delta_8k < -1.0 and delta_4k < 1.0:
    print("✓ SUCCESS: 8K PPL improved, 4K PPL preserved. Run full 1000 steps on 4090.")
elif delta_8k < 0:
    print("~ PARTIAL: 8K PPL improving but slowly. More steps needed — run on 4090.")
elif delta_4k > 2.0:
    print("✗ WARNING: 4K PPL degraded significantly. Check YaRN patch and lr.")
else:
    print("? INCONCLUSIVE: Need more steps. Run on 4090.")

## 10. Loss curve

In [ ]:
import matplotlib.pyplot as plt

steps   = log['step']
loss_4k = log['loss_4k']
loss_8k = log['loss_8k']
phases  = log['phase']

fig, ax = plt.subplots(figsize=(12, 5))

# Shade phase regions
phase2_start = next((s for s, p in zip(steps, phases) if p == 2), None)
if phase2_start:
    ax.axvspan(phase2_start, steps[-1], alpha=0.08, color='orange', label='Phase 2 (unfrozen)')

ax.plot(steps, loss_4k, 'b-o', markersize=4, label='Loss @ 4K tokens', linewidth=2)
ax.plot(steps, loss_8k, 'r-o', markersize=4, label='Loss @ 8K tokens', linewidth=2)
ax.axhline(loss_4k_base, color='b', linestyle='--', alpha=0.4, label=f'Baseline 4K ({loss_4k_base:.4f})')
ax.axhline(loss_8k_base, color='r', linestyle='--', alpha=0.4, label=f'Baseline 8K ({loss_8k_base:.4f})')

ax.set_xlabel('Training Step')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('BitNet b1.58 YaRN Context Extension — 4K vs 8K Loss')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/loss_curve.png', dpi=150)
plt.show()
print("Saved: /content/loss_curve.png")

## 11. (Optional) Save final checkpoint to Google Drive

In [ ]:
# Uncomment to mount Drive and save
# from google.colab import drive
# drive.mount('/content/drive')
# 
# save_path = '/content/drive/MyDrive/bitnet_yarn_8k_200steps'
# model.save_pretrained(save_path)
# tokenizer.save_pretrained(save_path)
# 
# import shutil
# shutil.copy('/content/loss_curve.png', '/content/drive/MyDrive/loss_curve_yarn_200steps.png')
# print(f"Saved to Drive: {save_path}")

print("Uncomment the cells above to save to Google Drive.")
print("Checkpoint at step 100 and 200 are already saved to /content/")